In [2]:
import torch
import pandas as pd
import numpy as np

from utils.basic_classes import DataSet, StrategicModelParameters

data = DataSet.CORA.get_dataset(num_layers=1)
data
# prune_adjacency_real(edge_index, x, y, train_mask, method, k)

Data(x=[2708, 1433], edge_index=[2, 5278], y=[2708], train_mask=[2708], test_mask=[2708], num_classes=2)

In [3]:
import torch
import numpy as np

class PyTorchLTModel:
    def __init__(self, edge_index, num_nodes, weight_type='uniform', device='cpu'):
        """
        weight_type: 'uniform' or 'pagerank'
        """
        self.device = device
        self.num_nodes = num_nodes
        self.weight_type = weight_type
        
        # --- Preprocessing for Fast Traversal ---
        # 1. Sort edges by target (row 1) to optimize for incoming edge lookups
        src, dst = edge_index[0], edge_index[1]
        sort_idx = torch.argsort(dst)
        self.src_sorted = src[sort_idx].to(device)
        self.dst_sorted = dst[sort_idx].to(device)
        
        # 2. Compute Degree and Pointers (CSR format)
        self.degree = torch.bincount(self.dst_sorted, minlength=num_nodes)
        self.ptr = torch.cat([torch.tensor([0], device=device), 
                              torch.cumsum(self.degree, 0)])

        # 3. Compute Edge Weights (Stored for reference or advanced sampling)
        if weight_type == 'pagerank':
            self.weights = self._compute_pagerank_weights(self.src_sorted, self.dst_sorted, num_nodes)
        else:
            self.weights = None

    def _compute_pagerank_weights(self, src, dst, num_nodes):
        """
        Computes w_uv based on the centrality (Out-Degree) of the influencer u.
        """
        # Calculate raw centrality scores (Out-Degree of the influencer)
        out_degree = torch.bincount(src, minlength=num_nodes).float()
        out_degree[out_degree == 0] = 1.0 
        
        # Assign score to each edge based on its source
        edge_scores = out_degree[src]
        
        # Sum incoming scores for each destination to normalize
        sum_scores = torch.zeros(num_nodes, device=self.device)
        sum_scores.scatter_add_(0, dst, edge_scores)
        
        # Normalize: w_uv = Score(u) / SumScores(v)
        denominators = sum_scores[dst]
        denominators[denominators == 0] = 1.0
        
        weights = edge_scores / denominators
        return weights

    def generate_rr_sets_vectorized(self, num_samples, max_depth=50):
        current_nodes = torch.randint(0, self.num_nodes, (num_samples,), device=self.device)
        traces = [current_nodes]
        active_mask = torch.ones(num_samples, dtype=torch.bool, device=self.device)
        
        for _ in range(max_depth):
            if not active_mask.any(): break
            
            # --- Vectorized Neighbor Sampling ---
            active_indices = torch.where(active_mask)[0]
            active_curr = current_nodes[active_indices]
            
            # Get degrees and verify valid moves
            degs = self.degree[active_curr]
            can_move = degs > 0
            
            if not can_move.any(): break
            
            # Filter for walkers that can actually move
            moving_indices = active_indices[can_move]
            moving_nodes = active_curr[can_move]
            moving_degs = degs[can_move]
            
            # --- Selection Logic ---
            starts = self.ptr[moving_nodes]
            
            # fast uniform sampling: random offset [0, degree)
            # (Approximating weighted walk with structural walk for performance)
            offsets = (torch.rand(len(moving_nodes), device=self.device) * moving_degs).long()
            next_indices = starts + offsets

            next_nodes = self.src_sorted[next_indices]
            
            current_nodes[moving_indices] = next_nodes
            active_mask[:] = False
            active_mask[moving_indices] = True
            traces.append(current_nodes.clone())

        traces_stacked = torch.stack(traces, dim=1).cpu().numpy()
        rr_sets = [set(traces_stacked[i]) for i in range(num_samples)]
        return rr_sets

    def select_seeds_greedy(self, rr_sets, k):
        seeds = []
        for _ in range(k):
            counts = {}
            for rr_set in rr_sets:
                for node in rr_set:
                    counts[node] = counts.get(node, 0) + 1
            if not counts: break
            best_node = max(counts, key=counts.get)
            seeds.append(best_node)
            rr_sets = [s for s in rr_sets if best_node not in s]
        return seeds

# --- Usage ---
edge_index = data.edge_index
num_nodes = data.num_nodes

model = PyTorchLTModel(edge_index, num_nodes, weight_type='pagerank')
rr_sets = model.generate_rr_sets_vectorized(num_samples=num_nodes)
seeds = model.select_seeds_greedy(rr_sets, k=5)

print("Selected Seeds:", seeds)

Selected Seeds: [37143, 34445, 14156, 5073, 25233]


In [5]:
edge_index = data.edge_index
num_nodes = data.num_nodes

model = PyTorchLTModel(edge_index, num_nodes, weight_type='uniform')
rr_sets = model.generate_rr_sets_vectorized(num_samples=num_nodes)
seeds = model.select_seeds_greedy(rr_sets, k=50)

print("Selected Seeds:", seeds)

Selected Seeds: [16072, 33630, 30993, 34257, 37143, 6491, 35555, 15525, 37059, 5073, 2495, 32789, 25233, 11802, 38309, 28511, 19267, 18527, 28644, 9553, 21241, 8866, 6219, 29379, 22241, 37585, 5600, 18209, 25729, 27788, 9452, 34009, 33098, 13491, 1373, 11914, 22222, 15227, 38165, 39247, 18416, 29143, 21804, 3943, 17370, 21366, 14301, 11132, 34638, 10489]
